# Hanny - M5 Kaggle runner (Hub-synced, resumable)

Import with **File > Import Notebook**. All training and evaluation logic lives
in the scripts in `hanny-code.zip`; this notebook only orchestrates them.

**What changed in this version:** checkpoints no longer live only in
`/kaggle/working`. `sft.py` pushes `adapter-last`, `adapter-best` and the
resume state (`train_state.pt`) to a **private** Hugging Face repo every 50
steps, and Cell 5 pulls them back at the start of the next session and resumes.
Kaggle wiping the session no longer costs you the run.

## One-time setup

1. Kaggle **Add-ons > Secrets**: add `HF_TOKEN` (a fine-grained Hugging Face
   token with **write** access to `leafstudiosDot/incotel-hanny-priv` only) and
   tick **Attached** for this notebook.
2. `hub_sync.py` must be in `hanny-code.zip`: add `"hub_sync.py"` to
   `CODE_FILES` in `pack.py`, rebuild, and upload a **new dataset version**.
   The patched `sft.py` (with `--resume`, `--save-every`, `--hub-repo`) must be
   in that same zip. Cell 2 rejects a stale version on purpose.
3. Notebook settings: **GPU T4 x2**, **Internet on**. Persistence does not
   matter any more; do not rely on it.

## Which path do you want?

| goal | cells to run |
| --- | --- |
| train / continue on both T4s with DDP (default) | 0 -> 1 -> 1b -> 2 -> 3 -> 4 (optional) -> 5 -> 6 -> 7 |
| train on one T4 (single process) | same, choose `4b` or `0.8b` in Cell 3 |
| gate-only, no retraining | 0 -> 1 -> 1b -> 2 -> 3 -> **5b** -> 7 (skip 4, 5, 6) |

Cell 5 decides by itself: nothing on the Hub -> fresh run; resume state on the
Hub -> `--resume`; run already finished -> skip training. Killing the session
mid-run and re-running the same cells is the supported way to continue.

- Do **not** install torch and do **not** `pip install -r requirements.txt`.
- Keep this notebook's output private: the gate result contains the private
  safety probes. Quote aggregate rates only, never the `per_probe` rows.
- `gate/` is never pushed to the Hub; only weights, resume state, `curve.csv`,
  `result.json` and `data_report.json` are.

### Cell 0 - GPU check (run this first)

This fails immediately if Kaggle did not actually enable the GPU, so you do not
wait through a dependency install only to hit a CPU trainer. `sft.py` refuses to
train on CPU; it cannot silently fall back.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

assert torch.cuda.is_available(), (
    "No GPU visible. In Kaggle use Session options > Accelerator > GPU T4 x2, "
    "save, restart the session, and re-run this cell. sft.py will not train on CPU.")
print("NOTE: Cell 3 selects the mode. DDP (default '4b-ddp') uses both T4s; "
      "the single-process path uses cuda:0 only.")

### Cell 1 - dependencies

Re-run in every new session. Kaggle already ships a CUDA-matched torch; the
pins below only add the training/gate stack around it.

In [ ]:
!pip install -q "transformers==5.20.0" "accelerate==1.15.0" "huggingface-hub==2.2.0" "peft==0.21.2" "bitsandbytes==0.50.2" "datasets==5.1.0"
!pip install -q flash-linear-attention || echo "fla unavailable -- reference path is fine"

from importlib.metadata import version

print({p: version(p) for p in
       ("transformers", "peft", "bitsandbytes", "datasets",
        "accelerate", "huggingface-hub")})
assert tuple(int(x) for x in version("bitsandbytes").split(".")[:2]) >= (0, 46)

### Cell 1b - Hugging Face secret and write check

Fails **now**, before any GPU time is spent, if the token is missing, the repo
is public, or the token cannot write. `hub_sync` deliberately never raises into
the training loop, so a token without write access would otherwise only show up
as a warning 50 steps in.

In [ ]:
import os, time
from huggingface_hub import HfApi

HUB_REPO = "leafstudiosDot/incotel-hanny-priv"   # PRIVATE: checkpoints + resume state
# leafstudiosDot/incotel-hanny is the PUBLIC repo. This notebook never writes to it;
# publish from your machine with upload.py (it enforces the rights/attribution gate).

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "No HF_TOKEN secret. Kaggle > Add-ons > Secrets > add HF_TOKEN, tick "
        "'Attached', then re-run this cell.") from e
os.environ["HANNY_HUB_REPO"] = HUB_REPO

api = HfApi(token=os.environ["HF_TOKEN"])
info = api.repo_info(HUB_REPO, repo_type="model")
assert info.private, f"{HUB_REPO} is PUBLIC. Make it private before training."
api.upload_file(
    path_or_fileobj=f"session started {time.strftime('%Y-%m-%d %H:%M:%S')} UTC\n".encode(),
    path_in_repo="_ping/last_session.txt", repo_id=HUB_REPO, repo_type="model",
    commit_message="session ping (write check)")
print(f"OK: {HUB_REPO} is private and writable.")
existing = sorted({f.split("/")[0] for f in api.list_repo_files(HUB_REPO, repo_type="model")
                   if "/" in f and not f.startswith("_ping")})
print("runs already on the Hub:", existing or "none")

### Cell 2 - unpack and verify

Unpacks both archives into `/kaggle/working`, plus any attached artifact bundle,
then refuses a stale dataset version before spending GPU time.

In [ ]:
import glob, os, shutil, subprocess, torch

ROOT = "/kaggle/working"
for stem in ("hanny-code", "hanny-data"):
    zips = glob.glob(f"/kaggle/input/**/{stem}.zip", recursive=True)
    dirs = [d for d in glob.glob(f"/kaggle/input/**/{stem}", recursive=True)
            if os.path.isdir(d)]
    if zips:
        print("unzip", zips[0])
        subprocess.run(["unzip", "-oq", zips[0], "-d", ROOT], check=True)
    elif dirs:
        print("copy", dirs[0])
        for name in os.listdir(dirs[0]):
            src, dst = os.path.join(dirs[0], name), os.path.join(ROOT, name)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
    else:
        raise RuntimeError(f"{stem}.zip or {stem}/ missing under /kaggle/input")

# Optional: a previously trained artifact bundle, for gate-only reruns.
for art in glob.glob("/kaggle/input/**/*artifacts.zip", recursive=True):
    print("unzip artifacts", art)
    subprocess.run(["unzip", "-oq", art, "-d", ROOT], check=True)

required = ["sft.py", "chat.py", "qwenkit.py", "chateval.py", "evalkit.py",
            "hub_sync.py", "evalsets/chat_probe.jsonl"]
missing = [r for r in required if not os.path.exists(os.path.join(ROOT, r))]
if missing:
    raise RuntimeError(f"missing {missing}; wrong dataset version? (hub_sync.py must be in "
                       "CODE_FILES in pack.py)")
print("safety probes:",
      "private" if os.path.exists(f"{ROOT}/evalsets/safety_probe.jsonl")
      else "PUBLIC FALLBACK (weaker gate)")
print("corpus:", glob.glob(f"{ROOT}/upload/corpus-*.json"))
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0),
      torch.cuda.get_device_capability(0))   # (7, 5) = T4 = no bf16
print("GPUs visible:", torch.cuda.device_count(),
      "(DDP uses both; single-process uses cuda:0 only)")
assert torch.cuda.is_available(), "Enable GPU T4 x2 in Notebook settings"

# Reject a stale dataset version before spending GPU time. These needles are the
# fixes that came out of the first 4B run: shifted eval loss, parent-memory
# release before the gate, a printable one-shot reply, thinking-off scoring,
# the system-prompt/UTF-8 provenance fixes, and the DDP path.
expected = {
    "sft.py": ["shift_logits", "empty_cache", "gate_exit", "gate_only",
               "utf-8-sig", "--ddp", "torch.distributed", "world_size", "local_sup",
               "write_resume_point", "--hub-repo", "--resume", "--save-every",
               # M6: restore-by-hash and the step-budget data cap. Without these
               # needles a pre-M6 hanny-code.zip passes here, then Cell 2b dies
               # with "unrecognized arguments" after paying for the model load.
               "pull_and_verify", "--expect-adapter-sha16", "--fit-steps",
               "val_per_example"],
    "qwenkit.py": ["cuda_index"],
    "chat.py": ["print(reply)"],
    "evalkit.py": ["enable_thinking", "system=None"],
    "chateval.py": ['"thinking": False', '"system": args.system',
                    "utf-8-sig", "chat_pairs_multiturn", "--check-probes"],
}
for fn, needles in expected.items():
    src = open(os.path.join(ROOT, fn), encoding="utf-8").read()
    missing = [n for n in needles if n not in src]
    if missing:
        raise RuntimeError(
            f"{fn} is stale (missing {missing}); upload the new dataset version")
# WP2: >=40 chat pairs is a claim threshold, not a preference. A zip built
# before WP2 passes every needle above and still ships 15 pairs, which would let
# the gate print a number it has no right to claim.
chat_probe_path = os.path.join(ROOT, "evalsets", "chat_probe.jsonl")
pairs = [ln for ln in open(chat_probe_path, encoding="utf-8")
         if ln.strip() and not ln.strip().startswith("//")]
n_multi = sum(1 for ln in pairs if '"history"' in ln)
if len(pairs) < 40:
    raise RuntimeError(
        f"evalsets/chat_probe.jsonl has {len(pairs)} pairs; the M6 gate needs at "
        "least 40 or every chat verdict is below the claim threshold. Rebuild "
        "hanny-code.zip (pack.py ships evalsets/ in CODE_FILES) and upload a new "
        "dataset version.")
print(f"chat probes: {len(pairs)} pairs ({n_multi} multi-turn)")
print("code version: shifted eval + gate cleanup + thinking off + DDP "
      "supervised-token scaling + Hub sync/resume + M6 restore/fit-steps + "
      "50-pair chat probes")

### Cell 2b - M6-A: restore from the Hub, prove the hash, talk to it

The restore half of "is `m5-4b-ddp` recoverable without Kaggle having kept the
file?". Three things are checked, in this order, because each one is what makes
the next meaningful:

1. the private repo serves `<run>/` to *this* token;
2. the weights that arrived hash to the expected adapter;
3. that adapter loads on `Qwen/Qwen3.5-4B` and answers through the chat template
   the run trained under.

`--pull` restores into `m6-restore/`, **not** into `checkpoints/`. Verifying what
the Hub holds and replacing your working copy are different actions, and an
adapter that arrived in an artifacts zip has already been mistaken for the Hub's
copy once.

`--pull-into DEST` puts the run at `DEST/<run>/` -- there is no `checkpoints/`
level inside `DEST` -- and `sft.py` now prints each verified adapter's full path.
The cell uses that layout rather than reconstructing it; the earlier version
assumed `DEST/checkpoints/<run>/` and failed *after* a successful, hash-verified
pull with `adapter path ... is not a directory`.

`--expect-adapter-sha16` is the assertion that matters. A path is a location, not
an identity: this run name has already held two different adapters, so "the pull
succeeded" is not "you restored the run you meant". On a mismatch `sft.py` exits
1 and prints both the expected and the found hash - that is a finding about the
Hub, not a bug in this cell.

**Expected** (`EXPECT_SHA16` is the only thing to edit below, and only if you
have deliberately decided the Hub is right):

```
pulling m5-4b-ddp from leafstudiosDot/incotel-hanny-priv into /kaggle/working/m6-restore/m5-4b-ddp/ ...
  train_state.pt present: True
  adapter-best  sha256[:16] de560f905163061b  (85.0 MB)  /kaggle/working/m6-restore/m5-4b-ddp/adapter-best
  adapter-last  sha256[:16] de560f905163061b  (85.0 MB)  /kaggle/working/m6-restore/m5-4b-ddp/adapter-last
  result.json   run 'm5-4b-ddp' | steps_done 250 | best_step 250 | loss_after 1.xxxxx | gate_exit 0
  train_state   step 250 | seconds ... | n_train ...
VERIFIED: m5-4b-ddp/adapter-best is sha256[:16] de560f905163061b, one of the expected adapters.

restore check exit=0  (0 = hash matched, 1 = mismatch, 2 = nothing pulled)

one-shot reply (greedy, temperature 0, seed 0), adapter .../m6-restore/m5-4b-ddp/adapter-best:
template ok: a4aee8afcf2e0711 | reply_tokens 40+
```

then the reply text and the provenance file.

**What would prove this failed**

- `no HF_TOKEN` / exit 2: the secret is not attached, so nothing was tested.
  Re-run Cell 1b first.
- `ADAPTER MISMATCH ... found: ...=039a77ffa53a4d42`: the Hub serves the *earlier*
  adapter under this run name. Every number attributed to `de560f9` would then be
  attached to `039a77ff`. Stop: push the intended weights, or change the expected
  hash knowingly and re-quote every number that depended on it.
- exit 0 but `adapter-best` and `adapter-last` disagree: the final push stopped
  between the two folders, so the resume state and the adapter may not belong
  together.
- `reply_tokens` 0, or a template other than `a4aee8afcf2e0711`: the adapter
  loaded but is not the one this run trained, or the template changed under it.
- `no adapter under .../m6-restore/m5-4b-ddp/`: the pull printed VERIFIED and the
  cell still cannot find the weights. That is a path bug in the caller, not a bad
  adapter -- read the paths `--pull` printed.
- A reply that is fluent but off-topic proves nothing either way. This cell checks
  identity, not quality; quality is Cell 6 and the M6 eval suite.


In [ ]:
import json, os, re, subprocess, sys

ROOT = "/kaggle/working"
REPO = os.environ.get("HANNY_HUB_REPO", "leafstudiosDot/incotel-hanny-priv")
RUN_RESTORE = "m5-4b-ddp"
EXPECT_SHA16 = "de560f905163061b"     # the adapter this run is supposed to be
EXPECT_TEMPLATE = "a4aee8afcf2e0711"  # the template it was trained under
DEST = f"{ROOT}/m6-restore"

rc = subprocess.run(
    [sys.executable, "sft.py", "--pull", "--run", RUN_RESTORE,
     "--hub-repo", REPO, "--pull-into", DEST,
     "--expect-adapter-sha16", EXPECT_SHA16],
    cwd=ROOT).returncode
print(f"\nrestore check exit={rc}  (0 = hash matched, 1 = mismatch, 2 = nothing pulled)")
if rc != 0:
    raise AssertionError(
        "The pull did not produce the expected weights. Read the hash table above "
        "before continuing: a restore that returns a different adapter is the one "
        "failure this cell exists to catch.")

adapter = f"{DEST}/{RUN_RESTORE}/adapter-best"
if not os.path.isdir(adapter):
    adapter = f"{DEST}/{RUN_RESTORE}/adapter-last"
if not os.path.isdir(adapter):
    raise AssertionError(
        f"no adapter under {DEST}/{RUN_RESTORE}/. --pull-into DEST restores the run "
        "at DEST/<run>/, not DEST/checkpoints/<run>/; the paths printed by --pull "
        "above are the ones to use.")
print(f"\none-shot reply (greedy, temperature 0, seed 0), adapter {adapter}:")
r = subprocess.run(
    [sys.executable, "chat.py", "--model", "Qwen/Qwen3.5-4B",
     "--adapter", adapter, "--temperature", "0", "--seed", "0",
     "--max-new-tokens", "64", "--json",
     "--ask", "In two sentences: what does a hash table do, and when is it the "
              "wrong choice?"],
    cwd=ROOT, capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    raise AssertionError(f"chat.py exited {r.returncode}")

cut = r.stdout.rfind("\n{")
if cut < 0:
    raise AssertionError("chat.py printed no JSON; the stdout above is all of it")
banner, data = r.stdout[:cut], json.loads(r.stdout[cut + 1:])
m = re.search(r"template ([0-9a-f]{16})", banner)
template = m.group(1) if m else None
print(f"template ok: {template} | reply_tokens {data['reply_tokens']} | "
      f"{data['seconds']}s | thinking {data['sampling']['thinking']}")
print("\nreply:\n" + data["reply"])

assert data["reply_tokens"] > 0, "the adapter loaded and generated nothing"
assert template == EXPECT_TEMPLATE, (
    f"template {template} != {EXPECT_TEMPLATE}: this adapter was trained under a "
    "different chat template, so its replies are not comparable with the run")

with open(f"{ROOT}/m6-restore-state.json", "w", encoding="utf-8") as f:
    json.dump({"run": RUN_RESTORE, "repo": REPO, "expected_sha16": EXPECT_SHA16,
               "adapter_path": adapter, "chat_template_sha16": template,
               "reply_tokens": data["reply_tokens"],
               "seconds": data["seconds"]}, f, indent=2)
print(f"\nwrote {ROOT}/m6-restore-state.json")


### Cell 2c - M6-B: does a killed-and-resumed run land where the uninterrupted one did?

Restore and `--resume` have never been exercised on real hardware. This is that
experiment, and it is deliberately small and deliberately *not* a training
result: ~1k conversations from 480 rows per source (`--fit-steps`), 120 steps,
gate off. Its only job is to show that two runs differing solely in the
interruption end at the same held-out loss.

Two runs of the same command:

- **A** - uninterrupted, 120 steps.
- **B** - same command. The cell watches B's log and SIGKILLs it the moment the
  first `[hub_sync] pushed ... step N` line appears, i.e. once that upload has
  committed. Then it deletes B's local run folder - so the Hub is the only
  remaining source - pulls B back, and resumes it to 120.

The comparison is `evalkit.paired_se` over `val_per_example`, the per-example
held-out losses behind `loss_after` that `sft.py` now stores for exactly this.
Pairing is only valid if both runs scored the same items, so the cell first
compares `loss_before` - the untouched 4-bit base on that same val set. Two
different val sets move that number by orders of magnitude more than 1e-4;
fp16 and kernel jitter do not.

Pass = `|A - B| <= 2 x paired se`, both runs reached 120 steps, and B really
resumed from the killed step. A run that stops short of 120 is a fail even if its
loss looks fine.

**Cost**: ~25 min on one T4 (about 10 for A, 6 to reach the kill point, 10 to
resume, plus three small data builds). Single process on purpose: DDP resume stays
untested after this cell, and must be reported as untested.

**Expected**

```
A: 120 optimizer steps, 3 evals ...
A exit=0
B pid ...
[kill] step 20 committed -> SIGKILL
B killed at step 20 (exit -9)
deleted /kaggle/working/checkpoints/m6-resume-b-... -- the Hub is now the only source
pulling m6-resume-b-... ...
  train_state.pt present: True
  adapter-last  sha256[:16] ...
pull exit=0
B: resume to completion
B resume exit=0

A  steps_done 120  best_step 120  loss_after 1.2xxx
B  steps_done 120  best_step 120  loss_after 1.2xxx  resumed_from_step 20
val examples scored: A 128, B 128
loss_before: A 1.4xxx  B 1.4xxx  gap 0.00e+00
  ok    same val set
  ok    same val count
  ok    same data cap
  ok    B really resumed
  ok    both reached the budget

paired change A-B +0.00xx nats, paired se 0.0xxx -> 2x se 0.0xxx
acceptance: PASS  (|A-B| <= 2x paired se)
```

**What would prove this failed**

- `cannot resume` / `this run's setup differs`: the streaming head was not stable
  between the two builds, so `n_train` changed. That is a real defect in the
  resume path, not bad luck - pin the data to a local snapshot and say so. Do not
  quote a resume result until it passes with a pinned dataset.
- The pull has no `train_state.pt` (exit 2): the Hub held no resumable state, so
  the "killed after a pushed save" premise was never met and nothing was tested.
- `resumed_from_step` missing or 0: B retrained instead of resuming. Its loss is
  then a different experiment, whatever it equals.
- `same val set` FAILs (`loss_before` gap above 1e-4): the two runs read
  different rows, so the paired se is meaningless. Do not report it.
- Both reached 120 but `best_step` differs while the losses are compared as if
  aligned: the pairing is between different steps; the cell prints a CAVEAT.
- The kill lands before any push (step 0): nothing was on the Hub, and the cell
  raises instead of passing.


In [ ]:
import json, os, re, shutil, subprocess, sys, time

ROOT = "/kaggle/working"
sys.path.insert(0, ROOT)
import evalkit

REPO = os.environ.get("HANNY_HUB_REPO", "leafstudiosDot/incotel-hanny-priv")
STEPS, EVAL_EVERY = 120, 40
STAMP = time.strftime("%H%M%S")
RUN_A, RUN_B = f"m6-resume-a-{STAMP}", f"m6-resume-b-{STAMP}"


def sft_cmd(run, resume=False):
    """One command shape for A and B. A resume has to repeat it exactly: the
    provenance check compares steps, world size, tokens/batch, seed and n_train."""
    cmd = [sys.executable, "-u", "sft.py", "--run", run,
           "--model", "Qwen/Qwen3.5-4B",
           "--from-hub", "allenai/tulu-3-sft-mixture",
           "--from-hub", "HuggingFaceH4/ultrachat_200k::default::train_sft",
           "--fit-steps", "--max-per-source", "3000",
           "--max-steps", str(STEPS), "--eval-every", str(EVAL_EVERY),
           "--val-per-mille", "100",
           "--max-len", "1024", "--max-tokens-per-batch", "2048",
           "--save-every", "20", "--no-gate", "--hub-repo", REPO]
    return cmd + (["--resume"] if resume else [])


for run in (RUN_A, RUN_B):
    assert not os.path.exists(f"{ROOT}/checkpoints/{run}"), (
        f"{run} already exists. This probe never overwrites a run: re-run the "
        "cell for a new timestamp.")
print(RUN_A, "|", RUN_B)

print("\nA: uninterrupted")
t0 = time.time()
rc = subprocess.run(sft_cmd(RUN_A), cwd=ROOT).returncode
print(f"A exit={rc}  {(time.time() - t0) / 60:.1f} min")
assert rc == 0, "A failed, so there is nothing to compare against."

print("\nB: start, then kill the moment a push commits")
log_b = f"{ROOT}/{RUN_B}.log"
with open(log_b, "w", encoding="utf-8") as f:
    pb = subprocess.Popen(sft_cmd(RUN_B), cwd=ROOT, stdout=f,
                          stderr=subprocess.STDOUT)
print(f"B pid {pb.pid}")
pushed_at, deadline = None, time.time() + 40 * 60
while time.time() < deadline and pb.poll() is None:
    time.sleep(5)
    try:
        with open(log_b, encoding="utf-8", errors="replace") as f:
            text = f.read()
    except OSError:
        continue
    m = re.search(rf"\[hub_sync\] pushed {re.escape(RUN_B)} step (\d+)", text)
    if m:
        pushed_at = int(m.group(1))
        break
if pushed_at is None:
    pb.kill()
    pb.wait()
    raise AssertionError(
        "No '[hub_sync] pushed ... step N' line before B exited or the deadline. "
        f"Read {log_b}: without a committed push there is nothing to resume from.")
print(f"[kill] step {pushed_at} committed -> SIGKILL")
try:
    pb.kill()
except ProcessLookupError:      # it exited between the log read and the kill
    pass
pb.wait()
print(f"B killed at step {pushed_at} (exit {pb.returncode})")
assert 0 < pushed_at < STEPS, "the kill landed outside the run; nothing mid-run to test"

local_b = f"{ROOT}/checkpoints/{RUN_B}"
assert os.path.isdir(local_b), "B never wrote a local run folder"
shutil.rmtree(local_b)
print(f"deleted {local_b} -- the Hub is now the only source")

rc = subprocess.run([sys.executable, "sft.py", "--pull", "--run", RUN_B,
                     "--hub-repo", REPO], cwd=ROOT).returncode
print(f"pull exit={rc}")
have_state = os.path.exists(f"{local_b}/train_state.pt")
print(f"train_state.pt restored: {have_state}")
assert rc == 0 and have_state, (
    "The Hub did not hold a resumable state for B, so the premise (killed after a "
    "pushed save) was not met. Nothing was tested.")

print("\nB: resume to completion")
t0 = time.time()
rc = subprocess.run(sft_cmd(RUN_B, resume=True), cwd=ROOT).returncode
print(f"B resume exit={rc}  {(time.time() - t0) / 60:.1f} min")
assert rc == 0, "the resumed run failed; read its output above"


def load(run):
    with open(f"{ROOT}/checkpoints/{run}/result.json", encoding="utf-8-sig") as f:
        return json.load(f)


a, b = load(RUN_A), load(RUN_B)
print(f"\nA  steps_done {a['steps_done']}  best_step {a['best_step']}  "
      f"loss_after {a['loss_after']}")
print(f"B  steps_done {b['steps_done']}  best_step {b['best_step']}  "
      f"loss_after {b['loss_after']}  resumed_from_step {b.get('resumed_from_step')}")
pa, pbb = a["val_per_example"], b["val_per_example"]
print(f"val examples scored: A {len(pa)}, B {len(pbb)}")
# --val-per-mille splits by a hash of each conversation, so the val set is the
# same set iff the same rows were read. loss_before is the untouched base model
# on that set, so a gap there means the pairing is invalid. 1e-4 is far below any
# real difference between two data sets and far above fp16/kernel jitter.
before_gap = abs(a["loss_before"] - b["loss_before"])
print(f"loss_before: A {a['loss_before']}  B {b['loss_before']}  gap {before_gap:.2e}")

checks = {
    "same val set": before_gap < 1e-4,
    "same val count": len(pa) == len(pbb) and len(pa) > 1,
    "same data cap": a.get("fit_steps") == b.get("fit_steps"),
    "B really resumed": b.get("resumed_from_step") == pushed_at,
    "both reached the budget": a["steps_done"] == b["steps_done"] == STEPS,
}
for name, ok in checks.items():
    print(f"  {'ok  ' if ok else 'FAIL'}  {name}")

if checks["same val set"] and checks["same val count"]:
    d, se = evalkit.paired_se(pa, pbb)
    within = se == se and abs(d) <= 2 * se
    print(f"\npaired change A-B {d:+.4f} nats, paired se {se:.4f} -> "
          f"2x se {2 * se:.4f}")
    print(f"acceptance: {'PASS' if within else 'FAIL'} "
          f"(|A-B| {'<=' if within else '>'} 2x paired se)")
    if a["best_step"] != b["best_step"]:
        print(f"CAVEAT: best steps differ (A {a['best_step']}, B {b['best_step']}); "
              "the losses are paired but were taken at different steps.")
    if not within:
        print("If |A-B| > 2x se, read B's log before blaming the resume: a data "
              "build that changed between the two runs shows up here.")
    if not all(checks.values()):
        print("NOTE: a check above failed, so a PASS here is not a clean resume "
              "result.")
else:
    print("\nnot paired: the two runs did not score the same items, so no paired "
          "se is reported. An empty val_per_example means the run predates the "
          "field (or had no best eval); re-run it with this sft.py.")


### Cell 3 - pick the run

`4b-ddp` is the default (both T4s, 250 steps). `4b-ddp-long` is the same recipe
with 1000 steps: change the string at the end of the dict to select it. `4b` is
the single-GPU v2 recipe; `0.8b` is the cheap pipeline validation.

**A resume needs the identical `steps`, seed, tokens per batch and world size**
as the interrupted run (`sft.py` checks and refuses otherwise). To go past a
*finished* run, make a new run name; do not raise `steps` on a finished one.
Runs are not comparable across `ddp`/`4b`/`0.8b` (different effective batch).

In [ ]:
CONFIG = {
    "0.8b":   dict(model="Qwen/Qwen3.5-0.8B", run="m5",         tokens=4096,
                   steps=800, ddp=False),
    "4b":     dict(model="Qwen/Qwen3.5-4B",   run="m5-4b-v2",   tokens=2048,
                   steps=500, ddp=False),
    "4b-ddp": dict(model="Qwen/Qwen3.5-4B",   run="m5-4b-ddp",  tokens=2048,
                   steps=250, ddp=True),
    # Same recipe, 4x the steps (about 3 h on 2 T4s; the 250-step run took 47 min and
    # saw ~1% of the data it built). New run name, so nothing is overwritten. With Hub
    # sync you can stop it at any time and re-run Cells 0-3 + 5 to continue.
    "4b-ddp-long": dict(model="Qwen/Qwen3.5-4B", run="m5-4b-ddp-long", tokens=2048,
                   steps=1000, ddp=True),
}["4b-ddp"]    # <-- "4b" for the single-GPU v2 recipe; "0.8b" for a cheap pass
MODEL, RUN = CONFIG["model"], CONFIG["run"]
DDP = bool(CONFIG.get("ddp"))
WORLD_SIZE = 2 if DDP else 1
STEPS = CONFIG["steps"]

# Cell 5 skips a run that already finished and resumes one that was interrupted.
# Set this True only when you deliberately mean to retrain from scratch (it then
# ignores resume state and overwrites checkpoints/<run> locally AND on the Hub).
ALLOW_RETRAIN = False
print(CONFIG, "| ddp", DDP, "| world_size", WORLD_SIZE)

### Cell 4 - optional data check (no weights, no GPU spend)

2000 rows per source is enough to see the mix and the supervised span. The
training cell streams the full cap.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "sft.py", "--dry-run", "--run", RUN + "-plan",
     "--model", MODEL,
     "--from-hub", "allenai/tulu-3-sft-mixture",
     "--from-hub", "HuggingFaceH4/ultrachat_200k::default::train_sft",
     "--max-per-source", "2000", "--max-len", "1024"],
    cwd="/kaggle/working", check=True)

### Cell 5 - train, resume, or skip (decides by itself)

1. Pulls `<run>/` from the private Hub repo into `checkpoints/` (no-op if empty).
2. If `result.json` says the run already finished `STEPS` steps -> skips training.
3. Else if `train_state.pt` is present -> adds `--resume`.
4. Else -> fresh run.

While it trains, rank 0 writes `adapter-last` + `train_state.pt` every 50 steps
and uploads them in a background thread (you will see `[hub_sync] pushed ...`
lines). The last step pushes the weights **before** the gate runs. A non-zero
exit means training or the gate failed; the adapter is on the Hub either way.

In [ ]:
import json, os, subprocess, sys, time, torch
sys.path.insert(0, "/kaggle/working")
import hub_sync

ROOT = "/kaggle/working"
run_dir = f"{ROOT}/checkpoints/{RUN}"
REPO = os.environ["HANNY_HUB_REPO"]

if DDP and torch.cuda.device_count() < WORLD_SIZE:
    raise RuntimeError(
        f"DDP needs {WORLD_SIZE} GPUs but only {torch.cuda.device_count()} are visible. "
        "Set Session options > Accelerator > GPU T4 x2, save, restart, and re-run Cell 0.")

sync = hub_sync.HubSync(REPO, RUN)
if not ALLOW_RETRAIN:
    sync.pull(f"{ROOT}/checkpoints")          # restores adapter-*/, train_state.pt, curve.csv, result.json

finished = False
rj = f"{run_dir}/result.json"
if os.path.exists(rj) and not ALLOW_RETRAIN:
    with open(rj, encoding="utf-8-sig") as f:
        finished = json.load(f).get("steps_done", 0) >= STEPS
resumable = os.path.exists(f"{run_dir}/train_state.pt") and not ALLOW_RETRAIN

if finished:
    print(f"{RUN}: already finished ({STEPS} steps). "
          "Skipping training. Cell 5b re-runs the gate if you want it.")
else:
    cmd = [
        sys.executable, "sft.py", "--run", RUN, "--model", MODEL,
        "--from-hub", "allenai/tulu-3-sft-mixture",
        "--from-hub", "HuggingFaceH4/ultrachat_200k::default::train_sft",
        "--max-per-source", "40000", "--epochs", "1",
        "--max-len", "1024",
        "--max-tokens-per-batch", str(CONFIG["tokens"]),
        "--max-steps", str(STEPS),
        "--eval-every", "100",
        "--save-every", "50",
        "--hub-repo", REPO,
    ]
    if resumable:
        cmd.append("--resume")
    if DDP:
        cmd = [sys.executable, "-m", "torch.distributed.run",
               "--standalone", f"--nproc_per_node={WORLD_SIZE}"] + cmd[1:] + ["--ddp"]
    print("MODE:", "RESUME" if resumable else "FRESH")
    print(" ".join(cmd))

    t0 = time.time()
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    rc = subprocess.run(cmd, cwd=ROOT, env=env).returncode
    print(f"sft.py exit={rc}  {(time.time()-t0)/60:.1f} min")

### Cell 5b - gate only, no retraining

Use this for a gate-only rerun (it pulls the adapter from the Hub if the session
does not have it) or after a gate failure. It is **skipped automatically** when
the run's gate already passed in this session, so *Run All* does not repeat the
gate that Cell 5 just ran; set `FORCE_GATE = True` to run it anyway. Single
process, no DDP, no training.

In [ ]:
import json, os, subprocess, sys
sys.path.insert(0, "/kaggle/working")
import hub_sync

FORCE_GATE = False
ROOT = "/kaggle/working"
run_dir = f"{ROOT}/checkpoints/{RUN}"
sync = hub_sync.HubSync(os.environ["HANNY_HUB_REPO"], RUN)

if not os.path.isdir(f"{run_dir}/adapter-best"):
    sync.pull(f"{ROOT}/checkpoints")
assert os.path.isdir(f"{run_dir}/adapter-best"), f"no adapter for {RUN} here or on the Hub"

already = False
rj = f"{run_dir}/result.json"
if os.path.exists(rj) and os.path.exists(f"{run_dir}/gate/result.json"):
    with open(rj, encoding="utf-8-sig") as f:
        already = json.load(f).get("gate_exit") == 0

if already and not FORCE_GATE:
    print(f"gate for {RUN} already ran and passed in this session; skipping (FORCE_GATE=True to repeat)")
else:
    rc = subprocess.run(
        [sys.executable, "sft.py", "--gate-only", "--run", RUN,
         "--model", MODEL, "--precision", "4-bit nf4"],
        cwd=ROOT).returncode
    print(f"gate-only exit={rc}")
    sync.push(run_dir, "gate", wait=True)     # result.json changed; gate/ itself is never pushed

### Cell 6 - one-shot chat smoke test

Kaggle has no stdin, so this is `--ask`, not the REPL. 160 tokens can end
mid-sentence; that is the cap, not a bug.

In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "chat.py", "--model", MODEL,
     "--adapter", f"checkpoints/{RUN}/adapter-best",
     "--ask", "Explain a hash table in three sentences, then give one Python example.",
     "--max-new-tokens", "160", "--json"],
    cwd="/kaggle/working", capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print("exit", r.returncode)
    print(r.stderr[-2000:])

### Cell 7 - results and artifact bundle

Prints the run summary and the gate summary with `per_probe` rows omitted, then
zips the whole run for download. Keep that zip private: it contains the gate
result, and that contains the private safety prompts.

In [ ]:
import json, os, zipfile

run_dir = f"/kaggle/working/checkpoints/{RUN}"
with open(os.path.join(run_dir, "result.json"), encoding="utf-8-sig") as f:
    r = json.load(f)
print(json.dumps(r, indent=2))

gate = os.path.join(run_dir, "gate", "result.json")
if os.path.exists(gate):
    with open(gate, encoding="utf-8-sig") as f:
        g = json.load(f)
    # Never print g["base"]["per_probe"] / g["tuned"]["per_probe"]: those rows
    # contain the private safety prompts. Aggregate numbers only.
    print("\nGATE (per-probe rows omitted):")
    keys = ("chat_loss_change", "chat_loss_change_paired",
            "chat_loss_change_paired_se", "balanced_behaviour_change",
            "refusal_discordant", "refusal_p_exact")
    print(json.dumps({k: g.get(k) for k in keys}, indent=2))
    for side in ("base", "tuned"):
        if side in g:
            print(side, {k: v for k, v in g[side].items() if k != "per_probe"})
else:
    print("\nNo gate result. Run Cell 5 or 5b before reading a verdict.")

out = f"/kaggle/working/{RUN}-artifacts.zip"
# train_state.pt (optimizer state, a few hundred MB) lives on the Hub for resuming;
# it does not belong in the download bundle.
SKIP = {"train_state.pt", "train_state.pt.tmp"}
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(run_dir):
        for name in files:
            if name.startswith(".") or name in SKIP:
                continue  # never ship .ddp_data.pt or other temporaries
            p = os.path.join(root, name)
            z.write(p, os.path.relpath(p, "/kaggle/working"))
with zipfile.ZipFile(out) as z:
    names = z.namelist()
assert any(n.endswith("adapter_model.safetensors") for n in names), "no adapter in zip"
print(f"\n{len(names)} files, {os.path.getsize(out)/1e6:.1f} MB -> {out}")

## Using both T4s

DDP is implemented in `sft.py` and selected by `4b-ddp` in Cell 3. It is
Kaggle-only by policy: `sft.py` refuses local training unless
`--allow-local-training` is passed for a deliberate smoke test. The local 4 GB
card cannot hold the 4B model plus optimizer state.

What the DDP path does:

- launches one process per T4 with
  `python -m torch.distributed.run --standalone --nproc_per_node=2`;
- builds the dataset once on rank 0 and shares the exact object, so the two
  ranks cannot disagree about batch counts;
- gives each rank disjoint batches and drops the remainder so every rank takes
  the same number of optimizer steps;
- scales each local loss by `local_supervised_tokens / global_supervised_tokens * world_size` before
  DDP's gradient average, so the update is weighted by global tokens rather
  than by rank;
- evaluates, saves and runs `chateval.py` on rank 0 only. Rank 1 waits at a
  barrier while the single-GPU gate runs, keeping the torchrun worker set alive.

It is a new run (`m5-4b-ddp`) with a different effective batch and a different
loss; it is not comparable with `m5-4b-v2`. `--grad-accum` must be 1. The first
2-GPU run is the validation of this path; a successful single-process smoke test
does not prove the multi-rank collectives, so read the first Kaggle DDP log
rather than assuming it worked.

Two **independent** experiments are also possible in two Kaggle sessions, or
with two subprocesses pinned by `CUDA_VISIBLE_DEVICES=0` and `=1`. That is two
results, not one faster run.

## Reading the result

- Quote the paired numbers: `chat_loss_change_paired` with
  `chat_loss_change_paired_se`, and the McNemar `refusal_p_exact`.
- Positive `chat_loss_change_paired` means the adapter's loss is lower. If
  `abs(change) < 2 * paired_se`, report it as **inconclusive**, not as a win.
- `"not distinguishable"` means these probes could not have seen the effect; it
  does not mean nothing happened.
- Do not quote `loss_after` as chat quality, and do not quote the old `m5-4b`
  run at all.
- Keep the artifact zip private; it contains the gate result, which contains the
  private safety prompts. Quote only aggregate rates.


## Checkpoints, resume and publishing

- Private checkpoints: `leafstudiosDot/incotel-hanny-priv`, one folder per run
  (`m5-4b-ddp/adapter-best`, `adapter-last`, `train_state.pt`, ...). The repo
  keeps every push as a commit, so it grows; squash its history occasionally
  from the repo's settings if it gets large.
- Stopped or crashed mid-run: start a new session, run Cells 0, 1, 1b, 2, 3,
  then 5. It prints `MODE: RESUME` and continues from the last 50-step save.
  Resume restores optimizer, LR schedule and data position; dropout masks
  differ, so it is statistically equivalent, not bit-identical.
- Public release: `leafstudiosDot/incotel-hanny` is never touched by this
  notebook. On your machine, pull the gated run and publish with the repo's own
  gate (it refuses private-data adapters and writes the attribution card):

  ```
  hf download leafstudiosDot/incotel-hanny-priv --include "m5-4b-ddp/*" --local-dir checkpoints
  .venv\Scripts\python.exe upload.py --adapter checkpoints/m5-4b-ddp/adapter-best --dry-run
  ```